In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Lab2-Transactions")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} — gotowy")

Spark 4.0.0-preview2 — gotowy


In [4]:
import json
import random
from datetime import datetime, timedelta

def generate_ecommerce_data(filename="transactions_10k.jsonl", count=10000):
    # Zakres czasu: dziś między 08:00 a 11:00
    start_time = datetime(2026, 5, 10, 8, 0, 0)
    total_seconds = 3 * 3600  # 3 godziny
    
    stores = ['SuperMarket', 'TechWorld', 'FashionHub', 'HomeDecor', 'BioShop']
    categories = ['Grocery', 'Electronics', 'Fashion', 'Home', 'Health']
    
    records = []
    
    for i in range(count):
        # Losowy czas w zadanym oknie
        random_offset = random.randint(0, total_seconds)
        ts = (start_time + timedelta(seconds=random_offset)).strftime('%Y-%m-%d %H:%M:%S')
        
        record = {
            "transaction_id": 100000 + i,
            "user_id": random.randint(100, 500), # Ograniczona pula użytkowników, by łatwiej testować anomalie
            "store": random.choice(stores),
            "category": random.choice(categories),
            "amount": round(random.uniform(10.0, 5500.0), 2),
            "timestamp": ts
        }
        records.append(record)
    
    # Sortowanie chronologiczne
    records.sort(key=lambda x: x['timestamp'])
    
    # Zapis do pliku JSONL
    with open(filename, 'w') as f:
        for r in records:
            f.write(json.dumps(r) + '\n')
            
    print(f"Pomyślnie wygenerowano {count} transakcji w pliku {filename}")

generate_ecommerce_data()

Pomyślnie wygenerowano 10000 transakcji w pliku transactions_10k.jsonl


In [6]:
df = spark.read.json("transactions_10k.jsonl")

print(f"Liczba rekordów: {df.count()}")
df.printSchema()


Liczba rekordów: 10000
root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- store: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- transaction_id: long (nullable = true)
 |-- user_id: long (nullable = true)



In [7]:
from pyspark.sql.functions import to_timestamp, col

df = df.withColumn("timestamp", to_timestamp(col("timestamp"), "yyyy-MM-dd HH:mm:ss"))

df.printSchema()  # timestamp powinien być teraz 'timestamp (nullable = true)'


root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- store: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- transaction_id: long (nullable = true)
 |-- user_id: long (nullable = true)



In [9]:
from pyspark.sql.functions import count, sum as _sum, avg, round as _round

store_summary = (
    df.groupBy("store")
    .agg(
        count("transaction_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
        _round(avg("amount"), 2).alias("srednia_PLN"),
    )
    .orderBy("store")
)
store_summary.show()

+-----------+---------+----------+-----------+
|      store|liczba_tx|  suma_PLN|srednia_PLN|
+-----------+---------+----------+-----------+
|    BioShop|     1956|5357181.81|    2738.85|
| FashionHub|     1995|5514430.98|    2764.13|
|  HomeDecor|     2012|5607615.32|    2787.09|
|SuperMarket|     2067|5806827.84|     2809.3|
|  TechWorld|     1970| 5593648.6|    2839.42|
+-----------+---------+----------+-----------+



In [10]:
from pyspark.sql.functions import sum as _sum, min as _min, max as _max

df.groupBy("category") \
    .agg(
        _sum("amount").alias("total_amount"),
        _min("amount").alias("min_amount"),
        _max("amount").alias("max_amount")
    ) \
    .orderBy("category") \
    .show()

+-----------+-----------------+----------+----------+
|   category|     total_amount|min_amount|max_amount|
+-----------+-----------------+----------+----------+
|Electronics|5584000.600000009|     11.34|   5496.64|
|    Fashion|5770776.619999995|     16.21|   5499.66|
|    Grocery|5371451.539999993|     12.44|   5499.23|
|     Health|5510411.409999988|      16.7|   5499.75|
|       Home|5643064.379999999|     10.29|    5499.2|
+-----------+-----------------+----------+----------+



In [12]:
from pyspark.sql.functions import window

hourly = (
    df.groupBy(window("timestamp", "1 hour"))    # okno 1-godzinne
    .agg(
        count("transaction_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .orderBy("window")
)
hourly.show(truncate=False)

+------------------------------------------+---------+----------+
|window                                    |liczba_tx|suma_PLN  |
+------------------------------------------+---------+----------+
|{2026-05-10 08:00:00, 2026-05-10 09:00:00}|3342     |9259139.85|
|{2026-05-10 09:00:00, 2026-05-10 10:00:00}|3302     |9231287.56|
|{2026-05-10 10:00:00, 2026-05-10 11:00:00}|3355     |9384657.13|
|{2026-05-10 11:00:00, 2026-05-10 12:00:00}|1        |4620.01   |
+------------------------------------------+---------+----------+



In [13]:
from pyspark.sql.functions import window

(
    hourly
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_tx",
        "suma_PLN",
    )
    .show(truncate=False)
)

hourly.show(truncate=False)

+-------------------+-------------------+---------+----------+
|od                 |do                 |liczba_tx|suma_PLN  |
+-------------------+-------------------+---------+----------+
|2026-05-10 08:00:00|2026-05-10 09:00:00|3342     |9259139.85|
|2026-05-10 09:00:00|2026-05-10 10:00:00|3302     |9231287.56|
|2026-05-10 10:00:00|2026-05-10 11:00:00|3355     |9384657.13|
|2026-05-10 11:00:00|2026-05-10 12:00:00|1        |4620.01   |
+-------------------+-------------------+---------+----------+

+------------------------------------------+---------+----------+
|window                                    |liczba_tx|suma_PLN  |
+------------------------------------------+---------+----------+
|{2026-05-10 08:00:00, 2026-05-10 09:00:00}|3342     |9259139.85|
|{2026-05-10 09:00:00, 2026-05-10 10:00:00}|3302     |9231287.56|
|{2026-05-10 10:00:00, 2026-05-10 11:00:00}|3355     |9384657.13|
|{2026-05-10 11:00:00, 2026-05-10 12:00:00}|1        |4620.01   |
+--------------------------------

In [14]:
from pyspark.sql.functions import window, count, sum as _sum

df.groupBy(window("timestamp", "30 minutes"), "store") \
    .agg(
        count("transaction_id").alias("transaction_count"),
        _sum("amount").alias("total_amount")
    ) \
    .orderBy("window", "store") \
    .show(truncate=False)

+------------------------------------------+-----------+-----------------+------------------+
|window                                    |store      |transaction_count|total_amount      |
+------------------------------------------+-----------+-----------------+------------------+
|{2026-05-10 08:00:00, 2026-05-10 08:30:00}|BioShop    |330              |874750.7699999997 |
|{2026-05-10 08:00:00, 2026-05-10 08:30:00}|FashionHub |359              |1005840.1900000004|
|{2026-05-10 08:00:00, 2026-05-10 08:30:00}|HomeDecor  |320              |953911.8299999991 |
|{2026-05-10 08:00:00, 2026-05-10 08:30:00}|SuperMarket|361              |963369.8400000007 |
|{2026-05-10 08:00:00, 2026-05-10 08:30:00}|TechWorld  |331              |906029.27         |
|{2026-05-10 08:30:00, 2026-05-10 09:00:00}|BioShop    |349              |997700.6200000003 |
|{2026-05-10 08:30:00, 2026-05-10 09:00:00}|FashionHub |325              |867131.9899999995 |
|{2026-05-10 08:30:00, 2026-05-10 09:00:00}|HomeDecor  |309 

In [16]:
from pyspark.sql.functions import window, sum as _sum, desc

df.filter(df.store == "TechWorld") \
    .groupBy(window("timestamp", "1 hour")) \
    .agg(_sum("amount").alias("total_revenue")) \
    .orderBy(desc("total_revenue")) \
    .show(1, truncate=False)

+------------------------------------------+------------------+
|window                                    |total_revenue     |
+------------------------------------------+------------------+
|{2026-05-10 10:00:00, 2026-05-10 11:00:00}|1881125.8200000008|
+------------------------------------------+------------------+
only showing top 1 row



In [18]:
sliding = (
    df.groupBy(window("timestamp", "1 hour", "30 minutes"))  # szerokość 1h, krok 30min
    .agg(
        count("transaction_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_tx",
        "suma_PLN",
    )
    .orderBy("od")
)
sliding.show(truncate=False)

+-------------------+-------------------+---------+----------+
|od                 |do                 |liczba_tx|suma_PLN  |
+-------------------+-------------------+---------+----------+
|2026-05-10 07:30:00|2026-05-10 08:30:00|1701     |4703901.9 |
|2026-05-10 08:00:00|2026-05-10 09:00:00|3342     |9259139.85|
|2026-05-10 08:30:00|2026-05-10 09:30:00|3326     |9233339.34|
|2026-05-10 09:00:00|2026-05-10 10:00:00|3302     |9231287.56|
|2026-05-10 09:30:00|2026-05-10 10:30:00|3294     |9275977.98|
|2026-05-10 10:00:00|2026-05-10 11:00:00|3355     |9384657.13|
|2026-05-10 10:30:00|2026-05-10 11:30:00|1679     |4666485.33|
|2026-05-10 11:00:00|2026-05-10 12:00:00|1        |4620.01   |
+-------------------+-------------------+---------+----------+



In [22]:
tumbling_rows = (
    df.groupBy(window("timestamp", "1 hour"))
    .agg(count("transaction_id"))
    .count()
)
sliding_rows = (
    df.groupBy(window("timestamp", "1 hour", "30 minutes"))
    .agg(count("transaction_id"))
    .count()
)
print(f"Tumbling (1h):          {tumbling_rows} okien")
print(f"Sliding  (1h / 30min):  {sliding_rows} okien")

# Odpowiedz w komentarzu: dlaczego sliding ma więcej wierszy?
# TWOJA ODPOWIEDŹ:
# Okno typu Sliding (przesuwne) generuje więcej wierszy, ponieważ okna nakładają się na siebie. 
# Nowe okno tumbling otwiera sie dopiero po zakończeniu poprzedniego, kiedy Sliding nakłada się na porzedni.

Tumbling (1h):          4 okien
Sliding  (1h / 30min):  8 okien


In [24]:
# Odpowiedz na pytania w komentarzach:

# 1. Ile transakcji jest w oknie 09:00–10:00?
#    Sprawdź w wyniku zadania 3.1.
#    ODPOWIEDŹ: 3302

# 2. Jaka jest różnica między groupBy("store") a groupBy(window(...), "store")?
#    ODPOWIEDŹ: groupBy("store") to agregacja globalna — otrzymujemy jeden wiersz na sklep 
#    dla całego zbioru danych
#    groupBy(window(...), "store") to agregacja czasowa — otrzymujemy wiele wierszy dla każdego sklepu, 
#    gdzie każdy wiersz reprezentuje osobny wycinek czasu

# 3. W oknie sliding 1h/30min — ile okien zawiera transakcje z godziny 09:30?
#    Wskazówka: narysuj oś czasu.
#    ODPOWIEDŹ: 2 okna; transakcja z godziny 09:30:00 wpada do okna [09:00 - 10:00] oraz do okna [09:30 - 10:30]

In [28]:
### PRACA DOMOWA

In [25]:
from pyspark.sql.functions import window, avg, asc

df.filter(df.store == "HomeDecor") \
    .groupBy(window("timestamp", "1 hour")) \
    .agg(avg("amount").alias("avg_amount")) \
    .orderBy(asc("avg_amount")) \
    .show(1, truncate=False)


+------------------------------------------+------------------+
|window                                    |avg_amount        |
+------------------------------------------+------------------+
|{2026-05-10 09:00:00, 2026-05-10 10:00:00}|2770.4079032258082|
+------------------------------------------+------------------+
only showing top 1 row



In [26]:
from pyspark.sql.functions import window, count

df.groupBy(window("timestamp", "30 minutes"), "category") \
    .agg(count("transaction_id").alias("tx_count")) \
    .filter("window.start = '2026-05-10 09:00:00'") \
    .orderBy("tx_count", ascending=False) \
    .show()

+--------------------+-----------+--------+
|              window|   category|tx_count|
+--------------------+-----------+--------+
|{2026-05-10 09:00...|Electronics|     356|
|{2026-05-10 09:00...|    Grocery|     345|
|{2026-05-10 09:00...|     Health|     340|
|{2026-05-10 09:00...|    Fashion|     324|
|{2026-05-10 09:00...|       Home|     320|
+--------------------+-----------+--------+



In [27]:
from pyspark.sql.functions import window, count, desc

df.groupBy(window("timestamp", "15 minutes")) \
    .agg(count("transaction_id").alias("total_transactions")) \
    .orderBy(desc("total_transactions")) \
    .show(1, truncate=False)


+------------------------------------------+------------------+
|window                                    |total_transactions|
+------------------------------------------+------------------+
|{2026-05-10 08:00:00, 2026-05-10 08:15:00}|866               |
+------------------------------------------+------------------+
only showing top 1 row

